In [1]:
energy_urls = {
    "Oil_Production_Barrels_Day": "https://www.globalfirepower.com/oil-production-by-country.php",
    "Oil_Consumption_Barrels_Day": "https://www.globalfirepower.com/oil-consumption-by-country.php",
    "Proven_Oil_Reserves_Barrels": "https://www.globalfirepower.com/proven-oil-reserves-by-country.php",
    "Natural_Gas_Production_Cubic_Meters": "https://www.globalfirepower.com/natural-gas-production-by-country.php",
    "Natural_Gas_Consumption_Cubic_Meters": "https://www.globalfirepower.com/natural-gas-consumption-by-country.php",
    "Proven_Natural_Gas_Reserves_Cubic_Meters": "https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php",
    "Coal_Production_Tonnes": "https://www.globalfirepower.com/coal-production-by-country.php",
    "Coal_Consumption_Tonnes": "https://www.globalfirepower.com/coal-consumption-by-country.php",
    "Proven_Coal_Reserves_Tonnes": "https://www.globalfirepower.com/proven-coal-reserves-by-country.php"
}

In [2]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [4]:
def scrape_metric(url, metric_name):

    print(f"Scraping {metric_name}...")

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed to load: {url}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    records = soup.find_all("div", class_="recordsetContainer")

    data = []

    for record in records:

        long_name = record.find("div", class_="longFormName")
        short_name = record.find("div", class_="shortFormName")
        value = record.find("div", class_="valueContainer")

        if value:

            if long_name:
                country = long_name.get_text(strip=True)

            elif short_name:
                country = short_name.get_text(strip=True)

            else:
                continue

            value = value.get_text(strip=True)

            data.append([country, value])

    df = pd.DataFrame(data, columns=["Country", metric_name])

    return df

In [5]:
dfs = []

for metric, url in energy_urls.items():

    df = scrape_metric(url, metric)

    if df is not None:
        dfs.append(df)

    time.sleep(2)

Scraping Oil_Production_Barrels_Day...
Scraping Oil_Consumption_Barrels_Day...
Scraping Proven_Oil_Reserves_Barrels...
Scraping Natural_Gas_Production_Cubic_Meters...
Scraping Natural_Gas_Consumption_Cubic_Meters...
Scraping Proven_Natural_Gas_Reserves_Cubic_Meters...
Scraping Coal_Production_Tonnes...
Scraping Coal_Consumption_Tonnes...
Scraping Proven_Coal_Reserves_Tonnes...


In [10]:
energy_df = dfs[0]

for df in dfs[1:]:

    energy_df = energy_df.merge(
        df,
        on="Country",
        how="outer"
    )

In [11]:
print(energy_df.head())

       Country                         Oil_Production_Barrels_Day  \
0  Afghanistan  0                        \t\t\t\t\t\t\r\n\t\t\...   
1      Albania  14,000                        \t\t\t\t\t\t\r\n...   
2      Algeria  1,443,000                        \t\t\t\t\t\t\...   
3       Angola  1,175,000                        \t\t\t\t\t\t\...   
4    Argentina  807,000                        \t\t\t\t\t\t\r\...   

                         Oil_Consumption_Barrels_Day  \
0  58,000                        \t\t\t\t\t\t\r\n...   
1  21,000                        \t\t\t\t\t\t\r\n...   
2  446,000                        \t\t\t\t\t\t\r\...   
3  121,000                        \t\t\t\t\t\t\r\...   
4  749,000                        \t\t\t\t\t\t\r\...   

                         Proven_Oil_Reserves_Barrels  \
0  0                        \t\t\t\t\t\t\r\n\t\t\...   
1  150,000,000                        \t\t\t\t\t\...   
2  12,200,000,000                        \t\t\t\t...   
3  7,783,000,000        

In [12]:
import re
import numpy as np

def clean_number(value):

    if pd.isna(value):
        return np.nan

    value = str(value)

    # Extract only the first numeric value
    match = re.search(r'[\d,]+', value)

    if match:
        number = match.group(0)
        number = number.replace(",", "")
        return int(number)

    return np.nan

In [13]:
for col in energy_df.columns[1:]:

    energy_df[col] = energy_df[col].apply(clean_number)

In [14]:
print(energy_df.info())

print(energy_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 10 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Country                                   145 non-null    object
 1   Oil_Production_Barrels_Day                145 non-null    int64 
 2   Oil_Consumption_Barrels_Day               145 non-null    int64 
 3   Proven_Oil_Reserves_Barrels               145 non-null    int64 
 4   Natural_Gas_Production_Cubic_Meters       145 non-null    int64 
 5   Natural_Gas_Consumption_Cubic_Meters      145 non-null    int64 
 6   Proven_Natural_Gas_Reserves_Cubic_Meters  145 non-null    int64 
 7   Coal_Production_Tonnes                    145 non-null    int64 
 8   Coal_Consumption_Tonnes                   145 non-null    int64 
 9   Proven_Coal_Reserves_Tonnes               145 non-null    int64 
dtypes: int64(9), object(1)
memory usage: 11.5+ KB
None

In [15]:
energy_df.to_csv("energy.csv", index=False)

print("Energy dataset saved successfully!")

Energy dataset saved successfully!
